In [ ]:
# 🤖 Milestone 2: Uncertainty Quantification for Robot Safety - CLEAN VERSION

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import os
import sys

# Add project path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.utils.preprocessing import load_fast_csv, window_data

# Set seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print("🤖 Uncertainty Quantification for Robot Safety - Clean Implementation")
print("="*70)

# ============================================================================
# 1. DATA LOADING & PREPROCESSING
# ============================================================================

def load_uncertainty_dataset(data_folder, n_files=5, window_size=512, samples_per_file=300):
    """Load and prepare dataset for uncertainty quantification"""
    
    files = [f for f in os.listdir(data_folder) if f.endswith("_fast.csv")][:n_files]
    print(f"Loading {len(files)} files for uncertainty analysis...")
    
    all_windows, all_rul = [], []
    
    for i, fname in enumerate(files):
        signal = load_fast_csv(os.path.join(data_folder, fname))
        windows = window_data(signal, window_size=window_size)[:samples_per_file]
        
        # Generate realistic RUL with degradation patterns
        max_rul = len(windows)
        rul = np.arange(max_rul, 0, -1).astype(float)
        
        # Add realistic measurement noise
        noise_level = 0.05 + 0.02 * i
        for j, window in enumerate(windows):
            noise = np.random.normal(0, noise_level, window.shape)
            windows[j] = window + noise
        
        all_windows.extend(windows)
        all_rul.extend(rul)
    
    X = np.array(all_windows)
    y = np.array(all_rul)
    
    # Normalize features
    X_reshaped = X.reshape(-1, 2)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_reshaped)
    X = X_scaled.reshape(X.shape)
    
    print(f"Dataset prepared: {X.shape[0]} samples, RUL range: {y.min():.1f}-{y.max():.1f}")
    return X, y, scaler

# ============================================================================
# 2. MONTE CARLO DROPOUT MODEL
# ============================================================================

def create_mc_dropout_model(input_shape, dropout_rate=0.25):
    """Create model with Monte Carlo Dropout for epistemic uncertainty"""
    
    return models.Sequential([
        layers.SeparableConv1D(12, 5, activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        layers.MaxPooling1D(2),
        
        layers.SeparableConv1D(8, 3, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        layers.GlobalAveragePooling1D(),
        
        layers.Dense(16, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(8, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(1)
    ])

class MCDropoutPredictor:
    """Monte Carlo Dropout uncertainty predictor"""
    
    def __init__(self, model, n_samples=100):
        self.model = model
        self.n_samples = n_samples
    
    def predict_with_uncertainty(self, X):
        """Generate predictions with uncertainty estimates"""
        
        predictions = []
        for _ in range(self.n_samples):
            pred = self.model(X, training=True)  # Keep dropout active
            predictions.append(pred.numpy().flatten())
        
        predictions = np.array(predictions)
        
        return {
            'mean': np.mean(predictions, axis=0),
            'std': np.std(predictions, axis=0),
            'lower_90': np.percentile(predictions, 5, axis=0),
            'upper_90': np.percentile(predictions, 95, axis=0),
            'lower_95': np.percentile(predictions, 2.5, axis=0),
            'upper_95': np.percentile(predictions, 97.5, axis=0)
        }

# ============================================================================
# 3. DEEP ENSEMBLE MODEL
# ============================================================================

class DeepEnsemble:
    """Deep ensemble for robust uncertainty quantification"""
    
    def __init__(self, n_models=5):
        self.n_models = n_models
        self.models = []
        self.trained = False
    
    def _create_diverse_model(self, input_shape, model_id):
        """Create architecturally diverse models for ensemble"""
        
        architectures = [
            # Standard architecture
            lambda: models.Sequential([
                layers.SeparableConv1D(12, 5, activation='relu', input_shape=input_shape),
                layers.BatchNormalization(),
                layers.MaxPooling1D(2),
                layers.SeparableConv1D(8, 3, activation='relu'),
                layers.GlobalAveragePooling1D(),
                layers.Dense(16, activation='relu'),
                layers.Dropout(0.2),
                layers.Dense(1)
            ]),
            # Wider architecture
            lambda: models.Sequential([
                layers.SeparableConv1D(16, 3, activation='relu', input_shape=input_shape),
                layers.BatchNormalization(),
                layers.MaxPooling1D(2),
                layers.SeparableConv1D(12, 5, activation='relu'),
                layers.GlobalAveragePooling1D(),
                layers.Dense(20, activation='relu'),
                layers.Dropout(0.15),
                layers.Dense(1)
            ]),
            # Deeper architecture
            lambda: models.Sequential([
                layers.SeparableConv1D(10, 3, activation='relu', input_shape=input_shape),
                layers.BatchNormalization(),
                layers.SeparableConv1D(8, 3, activation='relu'),
                layers.MaxPooling1D(2),
                layers.SeparableConv1D(6, 3, activation='relu'),
                layers.GlobalAveragePooling1D(),
                layers.Dense(12, activation='relu'),
                layers.Dense(8, activation='relu'),
                layers.Dropout(0.1),
                layers.Dense(1)
            ])
        ]
        
        return architectures[model_id % len(architectures)]()
    
    def fit(self, X_train, y_train, X_val, y_val, epochs=25):
        """Train ensemble of diverse models"""
        
        self.models = []
        learning_rates = [0.001, 0.002, 0.0015, 0.0008, 0.0012]
        
        for i in range(self.n_models):
            print(f"Training ensemble model {i+1}/{self.n_models}...")
            
            model = self._create_diverse_model(X_train.shape[1:], i)
            model.compile(
                optimizer=optimizers.Adam(learning_rate=learning_rates[i]),
                loss='mse',
                metrics=['mae']
            )
            
            # Train with different random seeds
            np.random.seed(42 + i)
            tf.random.set_seed(42 + i)
            
            model.fit(
                X_train, y_train,
                epochs=epochs,
                batch_size=32,
                validation_data=(X_val, y_val),
                verbose=0
            )
            
            self.models.append(model)
        
        self.trained = True
        print(f"✅ Ensemble training complete: {len(self.models)} models")
    
    def predict_with_uncertainty(self, X):
        """Generate ensemble predictions with uncertainty"""
        
        if not self.trained:
            raise ValueError("Ensemble must be trained first")
        
        predictions = []
        for model in self.models:
            pred = model.predict(X, verbose=0)
            predictions.append(pred.flatten())
        
        predictions = np.array(predictions)
        
        return {
            'mean': np.mean(predictions, axis=0),
            'std': np.std(predictions, axis=0),
            'lower_90': np.percentile(predictions, 5, axis=0),
            'upper_90': np.percentile(predictions, 95, axis=0),
            'lower_95': np.percentile(predictions, 2.5, axis=0),
            'upper_95': np.percentile(predictions, 97.5, axis=0)
        }

# ============================================================================
# 4. UNCERTAINTY CALIBRATION
# ============================================================================

def calculate_coverage(y_true, lower_bounds, upper_bounds):
    """Calculate coverage probability"""
    return np.mean((y_true >= lower_bounds) & (y_true <= upper_bounds))

def calibrate_uncertainty(results, y_true, target_coverage=0.9):
    """Calibrate uncertainty estimates for target coverage"""
    
    current_coverage = calculate_coverage(y_true, results['lower_90'], results['upper_90'])
    
    if current_coverage > 0.1:  # Avoid division by zero
        calibration_factor = target_coverage / current_coverage
        calibration_factor = np.clip(calibration_factor, 1.2, 3.0)
    else:
        calibration_factor = 2.5
    
    # Apply calibration
    mean_pred = results['mean']
    std_pred = results['std'] * calibration_factor
    
    return {
        'mean': mean_pred,
        'std': std_pred,
        'lower_90': mean_pred - 1.645 * std_pred,
        'upper_90': mean_pred + 1.645 * std_pred,
        'lower_95': mean_pred - 1.96 * std_pred,
        'upper_95': mean_pred + 1.96 * std_pred,
        'calibration_factor': calibration_factor
    }

# ============================================================================
# 5. ROBOT SAFETY INTEGRATION
# ============================================================================

class RobotSafetyPredictor:
    """Safety-aware predictor for robot fleet management"""
    
    def __init__(self, uncertainty_model, safety_threshold=0.75, conservative_factor=1.3):
        self.uncertainty_model = uncertainty_model
        self.safety_threshold = safety_threshold
        self.conservative_factor = conservative_factor
    
    def predict_with_safety(self, X):
        """Generate safety-aware predictions"""
        
        results = self.uncertainty_model.predict_with_uncertainty(X)
        
        predictions = results['mean']
        uncertainties = results['std']
        confidence_scores = 1 / (1 + uncertainties / np.mean(uncertainties))
        
        # Apply safety adjustments
        adjusted_predictions = []
        safety_levels = []
        
        for pred, conf in zip(predictions, confidence_scores):
            if conf >= self.safety_threshold:
                safety_level = "HIGH_CONFIDENCE"
                adjusted_pred = pred
            elif conf >= 0.5:
                safety_level = "MEDIUM_CONFIDENCE"
                adjusted_pred = pred / self.conservative_factor
            else:
                safety_level = "LOW_CONFIDENCE"
                adjusted_pred = pred / (self.conservative_factor * 1.5)
            
            adjusted_predictions.append(adjusted_pred)
            safety_levels.append(safety_level)
        
        return {
            'predictions': np.array(adjusted_predictions),
            'uncertainties': uncertainties,
            'confidence_scores': confidence_scores,
            'safety_levels': safety_levels,
            'confidence_intervals': (results['lower_90'], results['upper_90'])
        }

# ============================================================================
# 6. MAIN EXECUTION PIPELINE
# ============================================================================

def run_uncertainty_quantification(data_folder="../data/SMARTPDM_dataset.csv"):
    """Main pipeline for uncertainty quantification"""
    
    # 1. Load and prepare data
    X, y, scaler = load_uncertainty_dataset(data_folder)
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
    
    print(f"Data splits: Train={X_train.shape[0]}, Val={X_val.shape[0]}, Test={X_test.shape[0]}")
    
    # 2. Train MC Dropout model
    print("\n🎲 Training Monte Carlo Dropout model...")
    mc_model = create_mc_dropout_model(X_train.shape[1:])
    mc_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    mc_model.fit(X_train, y_train, epochs=25, batch_size=32, 
                 validation_data=(X_val, y_val), verbose=0)
    
    mc_predictor = MCDropoutPredictor(mc_model)
    mc_results = mc_predictor.predict_with_uncertainty(X_test)
    
    # 3. Train Deep Ensemble
    print("\n👥 Training Deep Ensemble...")
    ensemble = DeepEnsemble(n_models=5)
    ensemble.fit(X_train, y_train, X_val, y_val)
    ensemble_results = ensemble.predict_with_uncertainty(X_test)
    
    # 4. Calibrate uncertainty
    print("\n🔧 Calibrating uncertainty estimates...")
    calibrated_results = calibrate_uncertainty(ensemble_results, y_test)
    
    # 5. Safety integration
    print("\n🤖 Integrating with robot safety system...")
    safety_predictor = RobotSafetyPredictor(ensemble)
    safety_results = safety_predictor.predict_with_safety(X_test[:20])
    
    # 6. Results summary
    print("\n📊 RESULTS SUMMARY:")
    print(f"MC Dropout MAE: {mean_absolute_error(y_test, mc_results['mean']):.2f}")
    print(f"Deep Ensemble MAE: {mean_absolute_error(y_test, ensemble_results['mean']):.2f}")
    print(f"Coverage (90%): {calculate_coverage(y_test, calibrated_results['lower_90'], calibrated_results['upper_90'])*100:.1f}%")
    print(f"Calibration factor: {calibrated_results['calibration_factor']:.2f}")
    print(f"Safety assessment: {len(safety_results['safety_levels'])} robots evaluated")
    
    return {
        'models': {'mc_model': mc_model, 'ensemble': ensemble},
        'results': {'mc': mc_results, 'ensemble': ensemble_results, 'calibrated': calibrated_results},
        'safety': safety_results,
        'scaler': scaler
    }

# ============================================================================
# 7. EXECUTE IF RUN DIRECTLY
# ============================================================================

if __name__ == "__main__":
    results = run_uncertainty_quantification()
    print("\n✅ Uncertainty quantification pipeline complete!")